# 🟢 新手模式 | Beginner Mode

**不用写代码。** 拖动滑杆、点“运行实验”，看曲线和回放怎么变，再回答一个小问题，就完成一次探索。

**三步操作：**
1. 点菜单栏 Run ▶▶ Run All Cells（全部运行一次，等默认结果出来）；
2. 拖动下面的滑杆改一个关键参数；
3. 点 **▶ 运行实验**，看对比图和一句话解读。

**诚实说明：** 你看到的数字全部来自真实引擎（不是演示动画），和完整模式是同一口径。
新手模式只写 `labs/data/lite/` 下的完成记录，**不计入教师成绩册**；想拿实验成绩，请接着做对应的完整模式。

# B2 新手模式：绿灯给谁多一点？


## 直觉问题

一个十字路口，东西向每小时来 600 辆车，南北向只有 150 辆。周期 60 秒、黄灯各 3 秒，剩下 54 秒绿灯该怎么分？

- **均分（27/27）**：公平，但东西向的车要等很久。
- **全给东西向**：南北向的车又排起了长队。

本实验只让你调一个数：**东西向绿灯时长**（南北向自动 = 54 − 东西向）。

**你的任务在最后**：找到平均延误最小的那个绿灯时长。


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
sys.dont_write_bytecode = True

# 新手驱动：滑杆 -> 真实引擎 -> 对比图（引擎本身不在本本里，全部复用完整模式的同一批代码）
from common.lite_drivers import (
    plot_curves, plot_bars,
)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

LAST = {}  # 最近一次运行的结果（供“保存心得”格子使用）
print("✓ 新手驱动就绪：下面拖动滑杆、点运行即可")


In [ ]:
from common.lite_drivers import run_b2_lite, interpret_b2, save_lite_record

ew = widgets.IntSlider(
    value=27, min=15, max=45, step=1,
    description="东西向绿灯 (s)：", style={"description_width": "120px"},
    layout=widgets.Layout(width="70%"),
)
run_btn = widgets.Button(description="▶ 运行实验", button_style="success")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output(wait=True)
        print(f"正在跑真 SUMO 仿真（东西向 {ew.value}s / 南北向 {54 - ew.value}s）……请稍等")
        res = run_b2_lite(ew.value)
        f, s, p = res["fixed"], res["student"], res["proportional"]
        print(f"固定配时 27/27：平均延误 {f['avg_wait']}s，平均排队 {f['avg_queue']}辆，"
              f"完成 {f['n_done']} 辆")
        print(f"你的配时 {res['ew_green']}/{res['ns_green']}：平均延误 {s['avg_wait']}s，"
              f"平均排队 {s['avg_queue']}辆，完成 {s['n_done']} 辆")
        print(f"流量比例配时 43/11：平均延误 {p['avg_wait']}s，平均排队 {p['avg_queue']}辆")
        display(plot_bars(
            ["固定 27/27", f"你的 {res['ew_green']}/{res['ns_green']}", "流量比例 43/11"],
            [f["avg_wait"], s["avg_wait"], p["avg_wait"]],
            "Average delay (s/veh)", "Average delay: fixed vs yours vs proportional"))
        print("💡 " + interpret_b2(res))
        save_lite_record("B2", {"ew_green": res["ew_green"],
                                "ns_green": res["ns_green"]},
                         {"avg_wait": s["avg_wait"], "avg_queue": s["avg_queue"]})
        LAST.clear()
        LAST.update({"exp": "B2",
                     "params": {"ew_green": res["ew_green"],
                                "ns_green": res["ns_green"]},
                     "metrics": {"avg_wait": s["avg_wait"],
                                 "avg_queue": s["avg_queue"]}})

run_btn.on_click(on_run)
display(ew, run_btn, out)
on_run(None)  # 首次自动跑一次（默认 27/27，即现状）
print("首次运行已完成，记录摘要：", LAST.get("metrics"))


## 🎯 找区间小任务

拖动滑杆，回答下面两个问题（写在下一个格子的心得里）：

1. 平均延误最小的绿灯时长大约是多少？它和“按流量比例分”（43/11）接近吗？
2. 绿灯从最优值继续往东西向加，南北向的排队会发生什么变化？为什么“越多越好”不成立？


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from common.lite_drivers import save_lite_record

note = widgets.Textarea(
    value="",
    placeholder="写一句话：你找到了什么规律？（例如：阈值越大，刹车越早，但……）",
    description="我的心得：",
    layout=widgets.Layout(width="100%", height="80px"),
)
save_btn = widgets.Button(description="💾 保存心得与完成记录", button_style="info")
save_out = widgets.Output()

def _on_save(b):
    with save_out:
        save_out.clear_output(wait=True)
        if not LAST:
            print("请先运行一次实验（上一个格子的“运行实验”按钮），再保存。")
            return
        path = save_lite_record(LAST["exp"], LAST["params"], LAST["metrics"],
                                note=note.value)
        print(f"✅ 已保存：{path}")
        print("新手模式完成记录不进教师成绩册；完整模式的成绩请去完整模式 notebook 完成。")

save_btn.on_click(_on_save)
display(note, save_btn, save_out)


---

## 下一步 | Next

- 完成上面的“找区间”小任务后，把你的心得保存下来（上一个格子）。
- 想拿实验成绩、学完整算法：去做对应的完整模式（同目录下不带 `_lite` 的 notebook）。
- 新手模式的意义是先建立直觉：参数变了、结果往哪个方向变、变化有多大。

完整模式：`B2_traffic_signal_pro.ipynb`
